<font color='IndianRed'>
<font size=9> 

**Curso Inferencia Estadística**

<font color = 'DodgerBlue'>
<font size = 5>
    
**Diagnósticos de regresión.**

<font size = 3>

Para que el modelo de **regresión lineal** sea fiable, en las conclusiones derivadas de las **estimaciones** de **intervalos** e **inferencias** (**intervalos de confianza**, **contrastes de hipótesis**,...) que realizamos a partir de dicho modelo se tienen que verificar cietas hipótesis.

Las tareas que realizan dichas verificaciones se denominan **diagnósticos de regresión**.

Estos se clasifican en tres categorías:

* Errores: los errores deben seguir una $N(0,\sigma)$ con la misma **varianza** y ser incorrelacionados.


* Modelo: los puntos se tienen que ajustar a la estructura lineal considerada.


* Observaciones anómalas: a veces unas cuántas observaciones no se ajustan al modelo y hay que detectarlas.


En general hay dos métodos: gráficos y numéricos. 

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Normalidad de los residuos.**

<font size = 3>

Para detectar la **normalidad** de los **residuos**, podemos aplicar todos los tests de **normalidad** vistos en el tema de **Bondad de Ajuste**:

* Test **Kolmogorov-Smirnov-Lilliefors**.

* Test **Anderson-Darling**.

* Test de **Shapiro-Wilks**.

* Test **D'Agostino-Pearson**.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Normalidad de los residuos en Python**

<font size = 3>
    
En **Python**, si ``modelo`` es el modelo, entonces ``modelo.resid`` es el vector de errores. Por lo tanto es a este a vector a quien le tenemos que hacer la prueba de gaussianidad. Recuerda que para utilizarlos en **Python** requieres la biblioteca ``scipy.stats`` y/o ``statsmodels``

$p$-valor|Decisión|Significado
:--|:--|:--
Pequeño|Rechazar $H_0$|Hay buena probabilidad de que **NO es gaussiana**
Grande|Rechazar $H_1$|Hay buena probabilidad de que **SÍ es gaussiana**

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Homocedasticidad.**

<font size = 3>
    
Para verificar la **homocedasticidad** de los residuos, podemos aplicar el **Test de Breuch-Pagan**. Su aplicación es bastante sencilla:

* Obtener los residuos $\{e_i\}_{i=1}^n$ de la **regresión lineal** inicial.


* Calcular el **coeficiente de determinación** $R^2$ de la **regresión lineal** de los $e_i^2$ respecto de las variables iniciales. Es decir, calcular el **coeficiente de determinación** del modelo siguiente:

$$E^2=\gamma_0+\gamma_1X_1+\cdots+\gamma_kX_k$$

Se busca escribir a los errores $E^2$ como **regresión lineal** de las columnas explicativas.

* Calcular el **estadístico** $X_0=nR^2$ el cual, suponiendo **homocedasticidad**, sigue una $\chi^2_k$.


* Calculamos el p-valor $P(\chi^2_k\ge X_0)$ con el significado usual.


La regla de decisión es 

$p$-valor|Decisión|Significado
:--|:--|:--
Pequeño|Rechazar $H_0$|Hay buena probabilidad de que los residuos **NO sean homocedásticos**
Grande|Rechazar $H_1$|Hay buena probabilidad de que los residuos **SÍ sean homocedásticos**

También se puede hacer una prueba gráfica: cuando se grafica la nube de puntos de los valores ajustados (es decir, las $\hat{y_i}$) vs los residuos, deberemos observar un **cielo estrellado** centrado en la recta $y=0$.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Homocedasticidad en Python**

<font size = 3>

Para la prueba gráfica:

```python
################################################################
########################    EN PYTHON    #######################
################################################################

from plotnine import *

#### Obtener los residuos
residuos = modelo.resid

#### Obtener los valores y_gorro
y_gorro = modelo.fittedvalues

#### Realizar el gráfico

var_errores = pd.DataFrame({"y_gorro":y_gorro, "residuos":residuos})

(ggplot(data=var_errores) + 
 geom_point(mapping=aes(x="y_gorro",y="residuos"),color="blue") +
 geom_hline(yintercept=0)
)

################################################################
```

Para la prueba de  Breuch-Pagan en Python, simplemente usamos ``het_breuschpagan`` de la paquetería ``statsmodels.stats.diagnostic``.

```python
################################################################
########################    EN PYTHON    #######################
################################################################

from statsmodels.stats.diagnostic import het_breuschpagan

# Realizar la prueba de Breusch-Pagan

bp_test = het_breuschpagan(residuos, X)

# Los resultados
bp_stat = bp_test[0]   # Estadístico de la prueba
bp_p_value = bp_test[1]  # p-valor asociado

print(f"Estadístico de Breusch-Pagan: {bp_stat}")
print(f"p-valor: {bp_p_value}")

# Interpretación
if bp_p_value > 0.05:
    print("No se rechaza la hipótesis nula: los errores son homocedásticos.")
else:
    print("Se rechaza la hipótesis nula: los errores no son homocedásticos.")

################################################################
```

donde ``X`` es la tabla formada por las columnas descriptivas con una columna de 1's añadida a la izquierda.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Autocorrelación de los residuos: Test de Durbin-Watson.**

<font size = 3>

Otra de las hipótesis que se deben verificar para que el **análisis de regresión** sea correcto es la incorrelación de los residuos.

La autocorrelación de los residuos puede ser de dos tipos:

* Positiva: un valor positivo (negativo) de un error genera una cadena de residuos positivos (negativos)

* Negativa: los residuos alternan signo.

Es decir, que la **correlación** entre los errores sea cero, por lo que son independientes entre sí. A menos que, las variables sean **gaussianas**. Si las variables son **gaussianas** la independencia y la **correlación** son lo mismo.

En general, si $U$ y $V$ son independientes, entonces su **correlación** $(U, V) = 0$. Pero al revés no: se puede tener **correlación** $(U, V) = 0$ pero no ser independientes.

Peeeeeero: si $U$ y $V$ son **gaussianas** con **correlación** $(U, V) = 0$, entonces sí son independientes.

Para comprobar que los residuos no presentan **correlación**, se puede aplicar el **Test de Durbin-Watson**.

Sean $\{e_i\}_{i=1}^n$ los residuos de la **regresión**. Tomemos $E_i$ y $E_{i-1}$ como las **variables aleatorias** de error y consideremos la **recta de regresión** de $E_i$ respecto de $E_{i-1}$. Esto es, consideremos el modelo $E_i=\beta_0+\beta_1E_{i-1}$.

Se plantea el contraste $$\left\{\begin{array}{l}\mathcal{H}_0:\beta_1=0\\\mathcal{H}_1:\beta_1\neq0\end{array}\right.$$ con el estadístico de contraste $$d=\frac{\sum_{i=2}^n(e_i-e_{i-1})^2}{\sum_{i=1}^ne_i^2}$$

El valor de este **estadístico** es aproximadamente $2(1-b_1)$ donde $b_1$ es una estimación de $\beta_1$. Si $\mathcal{H}_0$ es cierta, su distribución es la de una cierta combinación lineal de $\chi^2$.

La regla de decisión es

| Estadístico DW   | Interpretación                             |
|------------------|--------------------------------------------|
| DW < 1.5         | Posible autocorrelación positiva           |
| 1.5 <= DW <= 2.5 | No hay evidencia significativa de autocorrelación |
| DW > 2.5         | Posible autocorrelación negativa           |

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Diagnósticos del modelo: Aditividad y linealidad**

<font size = 3>

Sea $i$ fija y perturbemos la variable $X_i$, digamos con $x_i+h$. Entonces

$$
\begin{array}{rcl}
\boldsymbol{Y|_{x_1,...,x_i+h,...x_k}-Y|_{x_1,...,x_i,...x_k}}&\boldsymbol{=}&\boldsymbol{\beta_0+\beta_1x_1+...+\beta_i(x_i+h)+...+\beta_kx_k-(\beta_0+\beta_1x_1+...+\beta_ix_i+...+\beta_kx_k)}\\
&\boldsymbol{=}&\boldsymbol{\beta_ih}
\end{array}
$$


Cuando se plantea un **modelo lineal**, se suponen implícitamente las condiciones siguientes:

* Aditividad: la variación en la expresión en **negrita** es independiente de las variables que no se perturbaron.

* Linealidad: la variación en la expresión en **negrita** es la misma independientemente de qué variable se perturbó.

Lo que nos quiere decir esto es que si tienes un **dato** perturbado y al restarle los **datos** originales, en esa diferencia solo se debe obtener el **dato** con la perturbación.

Este paso se debe hacer para evitar errores en los estudios estadísticos.

<font color = 'DodgerBlue'>
<font size = 5>
    
**Aditividad.**

<font size = 3>

Podemos comprobar la aditividad con el **Test de Tukey** usando los llamados gráficos de residuos parciales. La idea principal es verificar que no haya interacción entre las variables independienes y así, cada una tendrá un efecto aditivo en el modelo.

Si existe la interacción, algunos de los términos cuadráticos tendrán peso en el modelo. Esta es la base del **Test de Tukey**.

Para aplicarlo, se siguen los siguientes pasos:

* Se obtienen los valores ajustados $\{\widehat{y_i}\}$ por la **regresión lineal** inicial.

* Se lleva a cabo una segunda **regresión lineal** incluyendo como nueva variable independiente los $\widehat{y_i}^2$. Sea $\beta$ el coeficiente de esta nueva variable.

* Se testea si la variable $\widehat{y}^2$ es significativa en la segunda regresión. Es decir, se realiza el contraste $$\left\{\begin{array}{l}\mathcal{H}_0:\beta=0\\\mathcal{H}_1:\beta\neq0\end{array}\right..$$

Si no podemos descartar $\mathcal{H}_0$, la variable de los $\widehat{y_i}^2$ no es significativa y el modelo es aditivo.

La regla de decisión es

$p$-valor|Decisión|Significado
:--|:--|:--
Pequeño|Rechazar $H_0$|Hay buena probabilidad de que el modelo **NO es aditivo**
Grande|Rechazar $H_1$|Hay buena probabilidad de que el modelo **SÍ es aditivo**

En el **Test de Tukey** se buscan p-valores grandes.

<font color = 'DodgerBlue'>
<font size = 5>
    
**Linealidad.**

<font size = 3>

Los gráficos de **residuos parciales** son una herramienta útil para detectar la **no linealidad** en una **regresión**. Se definen los residuos parciales $e_{ij}$ para la **variable independiente** $X_j$ como $$e_{ij}=e_i+b_jx_{ij},$$ 

donde $e_i$ es el residuo $i$-ésimo de la **regresión lineal**; $b_j$ es el **coeficiente** de $X_j$ en la **regresión** original y $x_{ij}$ es la **observación** $j$-ésima del individuo $i$-ésimo.

Los **residuos parciales** se dibujan contra los valores de $x_j$ y se hace su recta de **regresión**.

Si esta no se ajusta a la curva dada por una **regresión** no paramétrica suave (las **variables independientes** no están predeterminadas y se construyen con los **datos**), el modelo no es **lineal**.

**Básicamente, es observar cómo se comportó el error en cada una de las columnas, ya no el error cometido en general. Lo que nos interesa es que el error sean variables gaussianas.**

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Observaciones anómalas.**

<font size = 3>

Las **observaciones anómalas** pueden provocar que se malinterpreten patrones en el conjunto de **datos**. Además, puntos aislados pueden tener una gran influencia en el **modelo de regresión**, dando resultados completamente diferentes. Por ejemplo, pueden provocar que nuestro modelo no capture características importantes de los **datos**. 

Por ello, es importante detectarlas.

Existen tres tipos de observaciones anómalas:

* **Leverages.** son **observaciones** con un **valor anómalo** de las **variables de control** (variables explicativas). No tienen por qué afectar los **coeficientes de la regresión**. Lo importante es que no son peligrosos.

* **Outliers de regresión** son **observaciones** que tienen un valor anómalo de la variable $Y$ (variable objetivo), condicionado a los valores de sus variables independientes $X_i$. Tendrán un **residuo muy alto** pero no pueden afectar demasiado a los **coeficientes de la regresión**. No son tan graves.

* **Observaciones influyentes** son aquellas que tienen un **leverage alto** (tiene mucha anomalía en las columnas predictoras); son **outliers** de **regresión** y afectan fuertemente a la **regresión**. Estos son peligrosos porque si te afectan mucho.

<img src = "Img/im020.png">

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Leverages (apalancamiento)**

<font size = 3>

Para hallar las **observaciones** que son **leverages**, necesitamos la matriz Hat: $H=X(X^TX)^{-1}X^T$. Como $\widehat{y}=Hy$, tenemos que $\widehat{y_j}=h_{1j}y_1+h_{2j}y_2+...+h_{nj}y_n=\sum_{i=1}^nh_{ij}x_i$ para $j=1,...,n$.

Si $h_{ij}$ es grande, la **observación** $i$-ésima tiene un impacto sustancial en el valor predicho $j$-ésimo.

Se define el **leverage de la observación** $i$ como su valor hat: $h_i=\sum_{j=1}^nh_{ij}^2$.
hat
La regla de decisión que nos dirá si una observación tiene leverage grande (y por lo tanto debe ser considerada con cuidado) es cuando su valor hat es mayor que $2(k+1)/n$.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Outliers**

<font size = 3>

Para detectar **outliers**, se estudian los *residuos estunderizados*. Es una **variable aleatoria** tipo la **t-student**. No es la **t-student**, pero está muy relacionado.

Se definen como $$E_i^\ast=\frac{e_i}{\sqrt{(MSE)_i(1-h_i)}}$$ donde $e_i$ es el error cometido en la predicción $i$ y $(MSE)_i$ es el error cuadrático medio del **modelo lineal** sin considerar el renglón $i$ y se aplica el **Test de Bonferroni**, obteniendo lo que se conoce como $p$-valor ajustado.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Observaciones influyentes.**

<font size = 3>

El método que usaremos es la llamada **Distanca de Cook**. Esta se define como $$D_i=\frac{E_i^\ast}{k+1}\cdot\frac{h_i}{1-h_i}.$$

Las **observaciones influyentes** son aquellas cuya **distancia de Cook** es mayor a $4/(n-k-2)$.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Tratamiento de las observaciones anómalas.**

<font size = 3>

El tratamiento suele muy complejo. 

Nos debemos preguntar si se deben a errores en la entrada o recogida de los **datos** y si este es el caso, eliminarlas.

Pero también pueden explicar que no se ha considerado alguna **variable independiente** que afecta al conjunto de **observaciones anómalas**.

> ⚠️ **Nota importante: Las más peligrosas son las influyentes.**.

En el supuesto de que se determine que se pueden eliminar, se tienen que eliminar de una a una actualizando el modelo cada vez.

Una vez que detectas los valores atípicos en los renglones. Supongamos que tienes $m$ valores atípicos en los renglones $1, 7, 9, 10, 11$. Realizas el diagnóstico sin el renglón $1$, luego realizas el diagnóstico sin el renglón $7$, luego realizas el diagnóstico sin el renglón $9$, luego realizas el diagnóstico sin el renglón $10$, luego realizas el diagnóstico sin el renglón $11$. 

De cada uno de los diagnósticos calculas en $R^2$. Eliminas el renglón que me haya producido menor $R^2$. Con esto, estarías cazando al modelo más inútil y ya sabes que no puedes quitar todos de golpe. Del alguna forma es eliminar la anomalía más problemática.

---

<font color = 'lightseagreen'>
<font size = 5>
    
**Una curiosidad (Rob J Hyndman).**

<font color = 'midnightblue'>
<font size = 3>

Alguien me envió este correo electrónico hoy:

 *Uno de mis colegas dijo que usted dijo/escribió una vez que había encontrado muy pocos **valores atípicos** reales en su trabajo, y que normalmente los puntos de **datos** que "parecían ser **atípicos**" eran puntos de **datos** adecuados que no deberían haber sido tratados como **atípicos**. ¿Ha discutido esto por escrito? Si es así, me encantaría leerlo.*
 
Creo que nunca he dicho o escrito algo así, y veo muchos **valores atípicos** en **datos** reales. Pero he aconsejado no omitir **valores atípicos** aparentes.

A menudo, la parte más interesante de un conjunto de **datos** está en las **observaciones** inusuales o inesperadas, por lo que me opongo firmemente a la omisión automática de **valores atípicos**. El caso más famoso de eso es la no detección del agujero en la capa de ozono por parte de la NASA. La forma en que me contaron la historia fue que los **valores atípicos** se habían filtrado automáticamente de los **datos** obtenidos de Nimbus-7. Fue solo cuando el British Antarctic Survey observó el fenómeno a mediados de la década de 1980 que los científicos regresaron y descubrieron que el problema podría haberse detectado una década antes si la NASA no hubiera aplicado el filtrado automático de **valores atípicos**. De hecho, así también se contó la historia en la web de la NASA durante unos años. Pero en una carta al editor del boletín IMS, Pukelsheim (1990) explica que la realidad era más complicada. En la historia corregida, los científicos estaban investigando las **observaciones** inusuales para ver si eran genuinas o el resultado de un error instrumental, pero aún así no detectaron el problema hasta bastante tarde.

Pase lo que pase realmente, los **valores atípicos** deben investigarse, no omitirse. Trate de comprender qué causó que algunas **observaciones** fueran diferentes del grueso de las **observaciones**. Si comprende los motivos, estará en una mejor posición para juzgar si los puntos pueden eliminarse legítimamente del conjunto de **datos** o si acaba de descubrir algo nuevo e interesante. Nunca elimines un punto solo porque es raro.

---

<font color = 'Brown'>
<font size = 4>
    
**Notas**

<font color = 'DarkRed'>
<font size = 3>

- Para poder confiar en nuestra **regresión lineal** más nos vale obtener p-valores grandes.
- En la **regresión lineal**, los errores deben ser **gaussianos**.

> ⚠️ **Nota importante: Si no se cumplen los diagnósticos de regresión (por ejemplo, si no hay gaussianidad en los residuos) simplemente no podemos confiar en los intervalos de confianza ni en la significancia de los coeficientes de la regresión.**

> ⚠️ **Nota importante: Recordar que los coeficientes son los números ($\beta_{0}$, $\beta{1}$, $\beta_{2}$, ..., $\beta_{k}$)**. Donde $Y = \beta_{0} + \beta{1} \ast X + \beta_{2} \ast X + ... + \beta_{k} \ast X$

- La **gaussianidad** se tiene que verificar en los errores cometidos, no en las columnas de la tabla.
- Si el $R^{2}$ ajustado es muy próximo a 1, es porque no tienes porqué preocuparte, simplemente no puedes confiar en los **intervalos de confianza**, porque los **intervalos de confianza** se construyen a partir del **teorema del límite central**.
- Si los errores no siguen una **distribución gaussianidad**, entonces los **intervalos de confianza** que se construyan para ellos no van a tener sentido. Los **intervalos de confianza** que conocemos solamente sirven para **gaussianidad**.
- Los errores cometidos deben tener **homocedasticidad**.
- Los errores cometidos son **variables aleatorias**  porque dependen de los **datos**. Por lo tanto, se les puede calcular su **varianza**. Lo que hay que ver es que todos tengan la misma **varianza**.
- En la **prueba de homocedasticidad**, más nos vale que en el **Test de Breuch-Pagan** salga un p-valor grande.
- En toda **prueba de hipótesis** siempre va haber un **estadístico** y un p-valor.
- Siempre se busca el $R^{2}$ con el valor más pequeño.